In [ ]:
import os, glob
import numpy as np
import torch
from recbole.quick_start import load_data_and_model

# 2) Load latest checkpoint
ckpt = max(glob.glob(os.path.join("saved", "*.pth")), key=os.path.getmtime)
config, model, dataset, train_data, valid_data, test_data = load_data_and_model(ckpt)



In [ ]:
import os, glob, zipfile, urllib.request
import pandas as pd
import numpy as np

genre_names = [
    "unknown","Action","Adventure","Animation","Children","Comedy","Crime","Documentary",
    "Drama","Fantasy","FilmNoir","Horror","Musical","Mystery","Romance","SciFi",
    "Thriller","War","Western"
]

def cfg_value(cfg, key, default=None):
    try:
        return cfg[key] if key in cfg else default
    except Exception:
        return default

def try_find_u_item_locally(config):
    data_path = cfg_value(config, "data_path", "")
    ds_name   = cfg_value(config, "dataset", "")

    candidates = []
    if data_path and ds_name:
        candidates += glob.glob(os.path.join(data_path, ds_name, "**", "u.item"), recursive=True)
        candidates += glob.glob(os.path.join(data_path, "**", "u.item"), recursive=True)

    candidates += glob.glob(os.path.join("dataset", "**", "u.item"), recursive=True)
    candidates += glob.glob(os.path.join(".", "**", "u.item"), recursive=True)

    candidates = list(dict.fromkeys(candidates))
    return max(candidates, key=os.path.getmtime) if candidates else None

def download_and_extract_u_item(cache_dir="ml100k_raw"):
    os.makedirs(cache_dir, exist_ok=True)
    zip_path = os.path.join(cache_dir, "ml-100k.zip")
    extract_dir = os.path.join(cache_dir, "ml-100k")

    url = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"

    if not os.path.exists(zip_path):
        print("Downloading:", url)
        urllib.request.urlretrieve(url, zip_path)

    with zipfile.ZipFile(zip_path, "r") as z:
        # Extract only what we need
        members = [m for m in z.namelist() if m.endswith("ml-100k/u.item")]
        if not members:
            raise FileNotFoundError("u.item not found inside the downloaded ml-100k.zip")
        z.extract(members[0], path=cache_dir)

    u_item_path = os.path.join(cache_dir, "ml-100k", "u.item")
    if not os.path.exists(u_item_path):
        raise FileNotFoundError(f"Expected extracted path not found: {u_item_path}")
    return u_item_path

def load_movies_u_item(u_item_path):
    cols = ["movie_id","title","release_date","video_release_date","imdb_url"] + genre_names
    movies = pd.read_csv(
        u_item_path,
        sep="|",
        names=cols,
        encoding="latin-1",
        engine="python"
    )
    movies["release_date"] = pd.to_datetime(movies["release_date"], errors="coerce")
    for g in genre_names:
        movies[g] = pd.to_numeric(movies[g], errors="coerce").fillna(0).astype(int)
    return movies

u_item_path = try_find_u_item_locally(config)
if u_item_path is None:
    u_item_path = download_and_extract_u_item(cache_dir="ml100k_raw")

print("Using u.item:", u_item_path)

movies = load_movies_u_item(u_item_path)

iid = dataset.iid_field  # "item_id" for you

internal_ids = np.arange(dataset.item_num)                
tokens = dataset.id2token(iid, internal_ids)             
item_map = pd.DataFrame({"item_id_internal": internal_ids, "movie_id_token": tokens})
item_map["movie_id"] = pd.to_numeric(item_map["movie_id_token"], errors="coerce").astype("Int64")

item_meta = item_map.merge(movies, on="movie_id", how="left").set_index("item_id_internal")

# Save all metadata aligned to RecBole ids
item_meta.reset_index().to_csv("ml100k_item_metadata_aligned.csv", index=False)
print("Saved ml100k_item_metadata_aligned.csv")

Using u.item: ./ml100k_raw/ml-100k/u.item
Saved ml100k_item_metadata_aligned.csv


,movie_id_token,movie_id,title,release_date,video_release_date,imdb_url,unknown,Action,Adventure,Animation,...,Fantasy,FilmNoir,Horror,Musical,Mystery,Romance,SciFi,Thriller,War,Western
item_id_internal,,,,,,,,,,,,,,,,,,,,,
0,[PAD],<NA>,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,242,242,Kolya (1996),1997-01-24,NaN,http://us.imdb.com/M/title-exact?Kolya%20(1996),0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,302,302,L.A. Confidential (1997),1997-01-01,NaN,http://us.imdb.com/M/title-exact?L%2EA%2E+Conf...,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
3,377,377,Heavyweights (1994),1994-01-01,NaN,http://us.imdb.com/M/title-exact?Heavyweights%...,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,51,51,Legends of the Fall (1994),1994-01-01,NaN,http://us.imdb.com/M/title-exact?Legends%20of%...,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1678,1674,1674,Mamma Roma (1962),1962-01-01,NaN,http://us.imdb.com/M/title-exact?Mamma%20Roma%...,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1679,1640,1640,"Eighth Day, The (1996)",1996-11-01,NaN,"http://us.imdb.com/Title?Huiti%E8me+jour,+Le+(...",0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1680,1637,1637,Girls Town (1996),1996-08-23,NaN,http://us.imdb.com/M/title-exact?Girls%20Town%...,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [1]:
import ast
import pandas as pd

rows = []
with open("dataset/steam-merged/steam_games.json", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        rows.append(ast.literal_eval(line))

meta = pd.DataFrame(rows)
print(meta.head())


          publisher                                             genres  \
0         Kotoshiro      [Action, Casual, Indie, Simulation, Strategy]   
1  Making Fun, Inc.               [Free to Play, Indie, RPG, Strategy]   
2      Poolians.com  [Casual, Free to Play, Indie, Simulation, Sports]   
3              彼岸领域                        [Action, Adventure, Casual]   
4               NaN                                                NaN   

                  app_name                    title  \
0      Lost Summoner Kitty      Lost Summoner Kitty   
1                Ironbound                Ironbound   
2  Real Pool 3D - Poolians  Real Pool 3D - Poolians   
3                  弹炸人2222                  弹炸人2222   
4            Log Challenge                      NaN   

                                                 url release_date  \
0  http://store.steampowered.com/app/761140/Lost_...   2018-01-04   
1  http://store.steampowered.com/app/643980/Ironb...   2018-01-04   
2  http://store.s

In [5]:
meta.iloc[0]["genres"]

['Action', 'Casual', 'Indie', 'Simulation', 'Strategy']

In [9]:
from collections import Counter

meta["genres"] = meta["genres"].apply(
    lambda x: x if isinstance(x, list) else []
)


top3 = (
    meta["genres"]
    .explode()          # flatten lists
    .dropna()
    .value_counts()
    .head(3)
    .index
    .tolist()
)

print("Top 3 genres:", top3)



Top 3 genres: ['Indie', 'Action', 'Casual']


In [10]:
genre_sets = meta["genres"].map(set)

for g in top3:
    meta[g] = genre_sets.map(lambda s: int(g in s))


In [ ]:
meta['item_id_internal'] = range(1, len(meta) + 1)

In [ ]:
meta.to_csv("steam_meta.csv")

In [ ]:
top10_devs = (
    meta["developer"]
    .dropna()
    .astype(str)
    .value_counts()
    .head(10)
)



developer
Ubisoft - San Francisco       1259
SmiteWorks USA, LLC            813
Dovetail Games                 253
KOEI TECMO GAMES CO., LTD.     232
Paradox Development Studio     156
Capcom                         130
Ronimo Games                   123
Choice of Games                100
Musopia                         95
Stainless Games                 95
Name: count, dtype: int64


In [6]:
targets = [
    "Ubisoft - San Francisco",
    "SmiteWorks USA, LLC",
    "Capcom",
]

for dev in targets:
    meta[dev] = (meta["developer"] == dev).astype(int)


In [15]:
meta.to_csv("steam_meta_V2.csv")